# PyTorch Fundamentals: Single-Neuron Regression from Training to Inference

This notebook demonstrates the core PyTorch workflow by training a **single-neuron linear regression model** to estimate delivery time from distance.

The goal is not to build a complex model. It is to understand the mechanics that appear in nearly every PyTorch training workflow:

**Tensor → Model → Loss → Optimizer → Forward Pass → Backpropagation → Parameter Update → Inference**

### Learning objectives
By the end of this notebook, I can:
- Create and inspect PyTorch tensors.
- Build a simple neural network with `nn.Linear`.
- Define a loss function and optimizer.
- Implement a training loop.
- Inspect learned weights and bias.
- Make predictions using `torch.no_grad()`.
- Explain a key limitation of a purely linear model.

> **Note:** This notebook is an independent learning implementation inspired by concepts from the DeepLearning.AI *PyTorch for Deep Learning Professional Certificate*. The code and explanations are organized for personal study and portfolio documentation.

## 1. Imports and Reproducibility

PyTorch provides the tensor operations, neural-network components, and optimization tools needed for this example. Matplotlib is used for visualization.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

# Reproducible parameter initialization
torch.manual_seed(42)

print(f"PyTorch version: {torch.__version__}")

## 2. Create a Small Dataset

For this introductory example, the dataset is defined directly as tensors. This keeps the notebook self-contained and lets the focus stay on model training rather than data ingestion.

`X` contains delivery distances in miles, and `y` contains observed delivery times in minutes.

In [ ]:
# Input feature: delivery distance in miles
X = torch.tensor([
    [1.0],
    [2.0],
    [3.0],
    [4.0],
    [5.0],
    [6.0]
], dtype=torch.float32)

# Target: delivery time in minutes
y = torch.tensor([
    [7.1],
    [11.9],
    [17.2],
    [21.8],
    [27.1],
    [32.0]
], dtype=torch.float32)

print("X shape:", X.shape)
print("y shape:", y.shape)

## 3. Build the Model

A single `nn.Linear(1, 1)` layer has:
- one input feature,
- one output value,
- one learnable weight,
- one learnable bias.

Mathematically, the model is:

\[
\hat{y} = wx + b
\]

where `w` and `b` are learned during training.

In [ ]:
model = nn.Sequential(
    nn.Linear(in_features=1, out_features=1)
)

print(model)

### Inspect the Initial Parameters

PyTorch initializes the weight and bias before training. These values are not yet meaningful—they are simply the model's starting point.

In [ ]:
layer = model[0]

print(f"Initial weight: {layer.weight.item():.4f}")
print(f"Initial bias:   {layer.bias.item():.4f}")

## 4. Define the Loss Function and Optimizer

**Mean Squared Error (MSE)** measures the difference between predicted and actual delivery times.

**Stochastic Gradient Descent (SGD)** updates the model parameters in the direction that reduces the loss.

In [ ]:
loss_fn = nn.MSELoss()
optimizer = optim.SGD(model.parameters(), lr=0.01)

## 5. Train the Model

Each training epoch follows the same fundamental sequence:

1. Clear old gradients.
2. Run a forward pass.
3. Compute the loss.
4. Run backpropagation.
5. Update model parameters.

This pattern is central to PyTorch model training.

In [ ]:
epochs = 500
loss_history = []

for epoch in range(epochs):
    # 1. Clear gradients from the previous iteration
    optimizer.zero_grad()

    # 2. Forward pass
    predictions = model(X)

    # 3. Compute loss
    loss = loss_fn(predictions, y)

    # 4. Backpropagation
    loss.backward()

    # 5. Update weight and bias
    optimizer.step()

    loss_history.append(loss.item())

    if (epoch + 1) % 100 == 0:
        print(f"Epoch {epoch + 1:3d} | Loss: {loss.item():.6f}")

## 6. Inspect What the Model Learned

The learned weight represents the estimated increase in delivery time for each additional mile. The bias represents the model's estimated baseline time when distance is zero.

In [ ]:
learned_weight = layer.weight.item()
learned_bias = layer.bias.item()

print(f"Learned weight: {learned_weight:.4f}")
print(f"Learned bias:   {learned_bias:.4f}")

print(
    f"\nLearned relationship:\n"
    f"Predicted time = {learned_weight:.4f} × distance + {learned_bias:.4f}"
)

## 7. Visualize Training Progress

A decreasing loss indicates that the model is finding parameter values that better explain the training data.

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(range(1, epochs + 1), loss_history)
plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.title("Training Loss")
plt.grid(alpha=0.3)
plt.show()

## 8. Visualize the Learned Relationship

The scatter points are the observed data. The line shows the predictions produced by the trained model.

In [ ]:
with torch.no_grad():
    fitted_y = model(X)

plt.figure(figsize=(7, 4))
plt.scatter(X.numpy(), y.numpy(), label="Observed data")
plt.plot(X.numpy(), fitted_y.numpy(), label="Model prediction")
plt.xlabel("Distance (miles)")
plt.ylabel("Delivery time (minutes)")
plt.title("Single-Neuron Linear Regression")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 9. Make an Unseen Prediction

During inference, gradients are unnecessary. `torch.no_grad()` disables gradient tracking and reduces unnecessary computation.

In [ ]:
distance_to_predict = 7.0

with torch.no_grad():
    new_distance = torch.tensor([[distance_to_predict]], dtype=torch.float32)
    predicted_time = model(new_distance).item()

print(
    f"Predicted delivery time for {distance_to_predict:.1f} miles: "
    f"{predicted_time:.2f} minutes"
)

## 10. Model-Capacity Experiment: What If the Pattern Is Nonlinear?

A single linear neuron can only represent a straight-line relationship. To illustrate this limitation, the next dataset follows a curved pattern.

The goal here is **not** to solve the nonlinear problem yet. It is to observe that training a linear model longer does not give the model the ability to represent a fundamentally different shape.

In [ ]:
nonlinear_X = torch.linspace(0, 6, 30).reshape(-1, 1)
nonlinear_y = 2.0 * nonlinear_X + 0.8 * nonlinear_X**2 + 3.0

linear_model = nn.Sequential(nn.Linear(1, 1))
nonlinear_loss_fn = nn.MSELoss()
nonlinear_optimizer = optim.SGD(linear_model.parameters(), lr=0.005)

for epoch in range(1000):
    nonlinear_optimizer.zero_grad()
    nonlinear_predictions = linear_model(nonlinear_X)
    nonlinear_loss = nonlinear_loss_fn(nonlinear_predictions, nonlinear_y)
    nonlinear_loss.backward()
    nonlinear_optimizer.step()

with torch.no_grad():
    final_linear_predictions = linear_model(nonlinear_X)

print(f"Final loss on nonlinear data: {nonlinear_loss.item():.4f}")

In [ ]:
plt.figure(figsize=(7, 4))
plt.scatter(
    nonlinear_X.numpy(),
    nonlinear_y.numpy(),
    label="Nonlinear data"
)
plt.plot(
    nonlinear_X.numpy(),
    final_linear_predictions.numpy(),
    label="Linear model"
)
plt.xlabel("Input")
plt.ylabel("Target")
plt.title("Limitation of a Single Linear Neuron")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 11. Key Takeaways

- PyTorch models operate on **tensors**.
- `nn.Linear(1, 1)` implements the equation \(\hat{y}=wx+b\).
- A **loss function** measures prediction error.
- `loss.backward()` computes gradients using backpropagation.
- The **optimizer** uses those gradients to update learnable parameters.
- `optimizer.zero_grad()` is required because PyTorch accumulates gradients by default.
- `torch.no_grad()` is appropriate when making predictions without training.
- More training cannot compensate for insufficient **model capacity**: a linear model remains linear regardless of the number of epochs.

### Core training pattern

```python
optimizer.zero_grad()
predictions = model(X)
loss = loss_fn(predictions, y)
loss.backward()
optimizer.step()
```

This basic pattern will reappear throughout more advanced PyTorch models.

## Next Step

The natural next step is to move beyond a single linear neuron and build a small multi-layer neural network capable of learning nonlinear relationships.